In [ ]:
import os
from pathlib import Path

# Set project root
PROJECT_ROOT = Path(r'C:\Users\User\Desktop\fake_news_project')

# Create necessary subdirectories if they don't exist
(PROJECT_ROOT / 'models').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'data').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'output').mkdir(parents=True, exist_ok=True)

# Models and data directories
models_dir = PROJECT_ROOT / 'models'
data_dir = PROJECT_ROOT / 'data'

# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import pickle
import scipy.sparse
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier
import warnings
from tqdm import tqdm
import matplotlib.pyplot as plt

# TensorFlow / Keras for BiLSTM
import tensorflow as tf

# Lime
import lime
from lime.lime_text import LimeTextExplainer

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

print("Project root set to:", PROJECT_ROOT)

## Text Preprocessing Pipeline

In [ ]:
# Load the combined dataset
df = pd.read_csv(data_dir / 'combined_fake_news_with_metadata.csv)
print("Original shape:", df.shape)
df.head()

In [ ]:
# Define preprocessing function
# Initialize stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
custom_stopwords = {'u', 'us', 's', 'amp', 'im', 'ive', 'dont', 'didnt', 'wasnt'}
stop_words = stop_words.union(custom_stopwords)
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    """
    Clean text: lower, remove punctuation/numbers, tokenize,
    remove stopwords, lemmatize.
    """
    if not isinstance(text, str):
        return ''
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation, numbers, and special characters (keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize
    words = text.split()
    # Remove stopwords and lemmatize
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    # Join back into a single string
    return ' '.join(words)

In [ ]:
# Apply preprocessing to the text colum
tqdm.pandas()
# Apply to the 'text' column
df['cleaned_text'] = df['text'].progress_apply(clean_text)

# Check a few samples
df[['text', 'cleaned_text']].head()

In [ ]:
# Add derived features
# Word count
df['word_count'] = df['cleaned_text'].apply(lambda x: len(x.split()))

# Character count
df['char_count'] = df['cleaned_text'].apply(lambda x: len(x))

# Exclamation marks count
df['exclamation_count'] = df['text'].apply(lambda x: str(x).count('!'))

# Show some stats
df[['label', 'word_count', 'char_count', 'exclamation_count']].describe()

In [ ]:
# Save to a new CSV
df.to_csv(data_dir / 'combined_fake_news_preprocessed.csv', index=False)
print(f"Saved preprocessed dataset to {output_path}")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

## TF‑IDF Feature Extraction

In [ ]:
# Load the preprocessed dataset
df = pd.read_csv(data_dir / 'combined_fake_news_preprocessed.csv')
print(df.shape)

In [ ]:
# Split into train/val/test
X = df['cleaned_text']
y = df['label']

# First split: 80% train+val, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: from the 80%, take 70% train, 10% val
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# Fit TF‑IDF vectorizer on training data only
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.7,
    stop_words='english'
)

# Fit on training data
with tqdm(total=1, desc="Fitting TF-IDF") as pbar:
    X_train_tfidf = tfidf.fit_transform(X_train)
    pbar.update(1)

# Transform validation and test data using the same vectorizer
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF shape: {X_train_tfidf.shape}")

In [ ]:
# View top words
# Get feature names
feature_names = tfidf.get_feature_names_out()

# Show top 20 words by TF-IDF score
avg_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).flatten()
top_indices = avg_tfidf.argsort()[-20:][::-1]
print("Top 20 words by average TF-IDF score:")
for idx in top_indices:
    print(f"{feature_names[idx]}: {avg_tfidf[idx]:.4f}")

In [ ]:
# Save the vectorizer and transformed data
# Save vectorizer
with open(models_dir / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Save sparse matrices
scipy.sparse.save_npz(data_dir / 'X_train_tfidf.npz', X_train_tfidf)
scipy.sparse.save_npz(data_dir / 'X_val_tfidf.npz', X_val_tfidf)
scipy.sparse.save_npz(data_dir / 'X_test_tfidf.npz', X_test_tfidf)

# Save labels as numpy arrays
np.save(data_dir / 'y_train.npy', y_train.values)
np.save(data_dir / 'y_val.npy', y_val.values)
np.save(data_dir / 'y_test.npy', y_test.values)

print("TF-IDF artifacts saved.")

## Traditional ML Model

In [ ]:
# Load the saved TF-IDF features and labels
X_train = scipy.sparse.load_npz(data_dir / 'X_train_tfidf.npz')
X_val = scipy.sparse.load_npz(data_dir / 'X_val_tfidf.npz')
X_test = scipy.sparse.load_npz(data_dir / 'X_test_tfidf.npz')

y_train = np.load(data_dir / 'y_train.npy')
y_val = np.load(data_dir / 'y_val.npy')
y_test = np.load(data_dir / 'y_test.npy')

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
# Define Evaluation Function (accuracy, precision, recall, f1)
def evaluate_model(model, X_val, y_val, X_test, y_test, model_name):
    # Validation
    y_val_pred = model.predict(X_val)
    val_acc = accuracy_score(y_val, y_val_pred)
    # Test
    y_test_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_prec = precision_score(y_test, y_test_pred)
    test_rec = recall_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    print(f"\n===== {model_name} =====")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Precision: {test_prec:.4f}")
    print(f"Test Recall: {test_rec:.4f}")
    print(f"Test F1-score: {test_f1:.4f}")
    print("\nClassification Report:\n", classification_report(y_test, y_test_pred, target_names=['Real', 'Fake']))

    return {
        'model': model_name,
        'val_acc': val_acc,
        'test_acc': test_acc,
        'precision': test_prec,
        'recall': test_rec,
        'f1': test_f1
    }

## ML Model Without Balances (Class Weight)

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

lr_results = evaluate_model(lr, X_val, y_val, X_test, y_test, "Logistic Regression")

# Save model
with open(models_dir / 'logistic_regression.pkl', 'wb') as f:
    pickle.dump(lr, f)

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_results = evaluate_model(rf, X_val, y_val, X_test, y_test, "Random Forest")

# Save model
with open(models_dir / 'random_forest.pkl', 'wb') as f:
    pickle.dump(rf, f)

In [ ]:
# XGBoost
xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)

xgb_results = evaluate_model(xgb, X_val, y_val, X_test, y_test, "XGBoost")

# Save model
with open(models_dir / 'xgboost.pkl', 'wb') as f:
    pickle.dump(xgb, f)

In [ ]:
# Compare Results
results = [lr_results, rf_results, xgb_results]
results_df = pd.DataFrame(results)
results_df = results_df.set_index('model')
print("Comparison of Traditional ML Models:")
print(results_df)

results_df[['test_acc', 'precision', 'recall', 'f1']].plot(kind='bar', figsize=(10,6))
plt.title('Performance Comparison of Traditional ML Models (Without Balance)')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## ML Model With Balances (Class Weight)

In [ ]:
# Logistic Regression
lr_bal = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_bal.fit(X_train, y_train)

lr_bal_results = evaluate_model(lr_bal, X_val, y_val, X_test, y_test, "Logistic Regression (balanced)")

# Save model
with open(models_dir / 'logistic_regression_balanced.pkl', 'wb') as f:
    pickle.dump(lr_bal, f)

In [ ]:
# Random Forest
rf_bal = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced', verbose=1) 
rf_bal.fit(X_train, y_train)

rf_bal_results = evaluate_model(rf_bal, X_val, y_val, X_test, y_test, "Random Forest (Balanced)")

# Save model
with open(models_dir / 'random_forest_balanced.pkl', 'wb') as f:
    pickle.dump(rf_bal, f)

In [ ]:
# XGBoost
# Compute scale_pos_weight based on training labels
pos_count = (y_train == 1).sum()
neg_count = (y_train == 0).sum()
scale_pos_weight = neg_count / pos_count
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

# Train XGBoost with balanced weighting
xgb_bal = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', scale_pos_weight=scale_pos_weight)
xgb_bal.fit(X_train, y_train, verbose=True)

xgb_bal_results = evaluate_model(xgb_bal, X_val, y_val, X_test, y_test, "XGBoost (balanced)")

# Save model
with open(models_dir / 'xgboost_balanced.pkl', 'wb') as f:
    pickle.dump(xgb_bal, f)

In [ ]:
# Compare Results
results = [lr_bal_results, rf_bal_results, xgb_bal_results]
results_df = pd.DataFrame(results)
results_df = results_df.set_index('model')
print("Comparison of Traditional ML Models:")
print(results_df)

results_df[['test_acc', 'precision', 'recall', 'f1']].plot(kind='bar', figsize=(10,6))
plt.title('Performance Comparison of Traditional ML Models (With Balance)')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Compare Results
results = [lr_results, lr_bal_results, rf_results, rf_bal_results, xgb_results, xgb_bal_results]
results_df = pd.DataFrame(results)
results_df = results_df.set_index('model')
print("Comparison of Traditional ML Models (with and without class balancing):")
print(results_df)

metrics = ['test_acc', 'precision', 'recall', 'f1']
results_df[metrics].plot(kind='bar', figsize=(12, 6))
plt.title('Performance Comparison (Balanced vs Original)')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Explainable AI (LIME)

In [ ]:
# Load tokenizer and define text cleaning function
# Load tokenizer
with open(os.path.join(models_dir, 'tokenizer.pkl'), 'rb') as f:
    tokenizer = pickle.load(f)

# Parameters must match training
MAX_SEQUENCE_LENGTH = 500

# Define text cleaning
stop_words = set(stopwords.words('english'))
custom_stopwords = {'u', 'us', 's', 'amp', 'im', 'ive', 'dont', 'didnt', 'wasnt'}
stop_words = stop_words.union(custom_stopwords)
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(words)

In [ ]:
# Load SavedModel and inspect signature
# Load SavedModel
saved_model_path = os.path.join(models_dir, 'bilstm_savedmodel')
model = tf.saved_model.load(saved_model_path)
infer = model.signatures['serve']

# Print signature to see input and output names
print(infer.structured_input_signature)
print(infer.structured_outputs)

In [ ]:
# Define prediction function for LIME
def predict_proba(texts):
    """
    LIME expects a function that takes a list of texts and returns a
    probability matrix of shape (n_samples, 2) where column 0 = prob(real),
    column 1 = prob(fake).
    """
    # Clean texts
    cleaned = [clean_text(t) for t in texts]
    # Tokenize
    sequences = tokenizer.texts_to_sequences(cleaned)
    # Pad to fixed length
    padded = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH,
                           padding='post', truncating='post')
    # Convert to float32 tensor
    input_tensor = tf.constant(padded, dtype=tf.float32)
    # Call the model
    outputs = infer(input_tensor)
    probs = outputs['output_0'].numpy()
    # Convert to two‑class probabilities
    return np.hstack([1 - probs, probs])

In [ ]:
# Load a few test examples from dataset
# Load preprocessed CSV (if have)
df = pd.read_csv(os.path.join(data_dir, 'combined_fake_news_preprocessed.csv'))
# Sample 5 random rows
samples = df.sample(5, random_state=42)
texts = samples['text'].tolist()
true_labels = samples['label'].tolist()
print("Sample texts:")
for i, t in enumerate(texts):
    print(f"{i}: {t[:100]}... (label: {true_labels[i]})")

In [ ]:
# Create LIME explainer and explain one instance
explainer = LimeTextExplainer(class_names=['Real', 'Fake'])

# Index 0 (first sample)
idx = 0
exp = explainer.explain_instance(
    texts[idx],
    predict_proba,
    num_features=10,
    top_labels=1
)

# Show explanation in notebook
exp.show_in_notebook(text=True)

# Save as HTML
exp.save_to_file(os.path.join(PROJECT_ROOT, 'lime_explanation_sample0.html'))
print(f"Explanation saved to {os.path.join(PROJECT_ROOT, 'lime_explanation_sample0.html')}")

In [ ]:
# Explain multiple samples
for idx in range(len(texts)):
    exp = explainer.explain_instance(texts[idx], predict_proba, num_features=10)
    print(f"\n=== Sample {idx} (true label: {true_labels[idx]}) ===")
    print(exp.as_list())
    # Save each as HTML
    exp.save_to_file(os.path.join(PROJECT_ROOT, f'lime_explanation_sample{idx}.html'))